In [1]:
import torch
torch.backends.cudnn.benchmark = True
from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
import evaluate
import os
os.environ["HF_AUDIO_BACKEND"] = "soundfile"
import torchaudio

e:\Projects\asrfinetune\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 1) Sanity recheck
import sys, platform
print("python:", sys.version.split()[0], "platform:", platform.platform())
print("device count:", torch.cuda.device_count())
print("visible devices:", os.environ.get("CUDA_VISIBLE_DEVICES"))

# 2) Smaller debug slice (fast first test)
small_train = dataset["train"].shuffle(seed=42).select(range(64))
small_eval = dataset["validation"].shuffle(seed=42).select(range(16))

# 3) safer training args for debugging
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-dysarthria-it",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=5e-6,
    warmup_steps=10,
    num_train_epochs=1,
    fp16=False,               # disable fp16 while debugging
    evaluation_strategy="steps",
    eval_steps=20,
    logging_steps=10,
    save_strategy="no",
    dataloader_num_workers=0, # avoid Windows worker hang
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to="none",
    load_best_model_at_end=False,
    predict_with_generate=False,  # speed check
)

# 4) small quick training run
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()


python: 3.10.11 platform: Windows-10-10.0.26200-SP0
device count: 1
visible devices: None


 31%|███▏      | 10/32 [00:02<00:05,  3.98it/s]

{'loss': 5.3211, 'grad_norm': inf, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.31}


 62%|██████▎   | 20/32 [00:05<00:03,  3.95it/s]

{'loss': 4.6203, 'grad_norm': 54.56214141845703, 'learning_rate': 4.0909090909090915e-06, 'epoch': 0.62}


TypeError: int() argument must be a string, a bytes-like object or a real number, not 'list'

In [2]:
import torch, transformers, datasets, evaluate
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), torch.cuda.device_count())
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("evaluate:", evaluate.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

torch: 2.5.1+cu121 cuda: True 1
transformers: 4.41.2
datasets: 4.8.4
evaluate: 0.4.6
device: cuda


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name())
else:
    print("CUDA not available. Training will be on CPU (slow).")

CUDA available: True
CUDA device count: 1
Current device: 0
Device name: NVIDIA GeForce RTX 4070 Laptop GPU


In [4]:
dataset = load_dataset("changelinglab/easycall-dysarthria")

print(dataset)
print(dataset["train"][0])

DatasetDict({
    test: Dataset({
        features: ['audio', 'filename', 'speaker', 'text', 'dysarthria_severity'],
        num_rows: 5213
    })
    train: Dataset({
        features: ['audio', 'filename', 'speaker', 'text', 'dysarthria_severity'],
        num_rows: 11901
    })
    validation: Dataset({
        features: ['audio', 'filename', 'speaker', 'text', 'dysarthria_severity'],
        num_rows: 4272
    })
})


RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.5.1+cu121) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call

In [5]:
from datasets import Audio
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000, decode=False))

In [6]:
processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small",
    language="italian",
    task="transcribe"
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [7]:
from datasets import disable_caching
disable_caching()

In [8]:
# def prepare(batch):
#     audio = batch["audio"]

#     # input features (log-Mel spectrogram)
#     batch["input_features"] = processor.feature_extractor(
#         audio["array"],
#         sampling_rate=audio["sampling_rate"]
#     ).input_features[0]

#     # labels (tokenized text)
#     batch["labels"] = processor.tokenizer(
#         batch["text"],
#         truncation=True
#     ).input_ids

#     return batch
import io, soundfile as sf

def prepare(batch, processor):
    audio = batch["audio"]

    if isinstance(audio, dict) and "array" in audio:
        waveform, sr = audio["array"], int(audio["sampling_rate"])
    elif isinstance(audio, dict) and "bytes" in audio:
        waveform, sr = sf.read(io.BytesIO(audio["bytes"]))
    elif isinstance(audio, dict) and "path" in audio:
        waveform, sr = sf.read(audio["path"])
    else:
        raise ValueError(f"Unsupported audio format: {audio}")

    # Resample to 16000 Hz if necessary
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000, dtype=torch.double)
        waveform = resampler(torch.tensor(waveform, dtype=torch.double).unsqueeze(0)).squeeze(0).numpy()
        sr = 16000

    batch["input_features"] = processor.feature_extractor(
        waveform, sampling_rate=sr
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"], truncation=True).input_ids
    return batch

In [9]:
dataset = dataset.map(
    prepare,
    remove_columns=["text", "speaker", "filename", "dysarthria_severity", "audio"],
    num_proc=0,
    fn_kwargs={"processor": processor},
)

Map: 100%|██████████| 4272/4272 [00:22<00:00, 187.65 examples/s]


In [10]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

# Enable memory optimization
model.gradient_checkpointing_enable()
model.config.use_cache = False

# Set language + task
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="italian",
    task="transcribe"
)

#  Move model to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
        

In [11]:
for param in model.model.encoder.parameters():
    param.requires_grad = False

In [12]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [13]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-dysarthria-it",

    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=5e-6,
    warmup_steps=100,

    num_train_epochs=10,

    fp16=True,

    evaluation_strategy="epoch",
    save_strategy="epoch",

    logging_steps=25,

    predict_with_generate=True,
    generation_max_length=128,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    
)

e:\Projects\asrfinetune\venv\lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [14]:
import setuptools
import distutils.version

In [15]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):

        # separate inputs and labels
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        # pad inputs
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        # pad labels
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        # replace padding with -100 for loss masking
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["input_ids"] == self.processor.tokenizer.pad_token_id,
            -100
        )

        batch["labels"] = labels

        return batch

In [16]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [17]:
from transformers import EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor.tokenizer,  # use tokenizer for text
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [18]:
print("train size", len(dataset["train"]))
print("val size", len(dataset["validation"]))
print("test size", len(dataset["test"]))

train size 11901
val size 4272
test size 5213


In [20]:
train_dataset=dataset["train"],          # Full 5,213 samples
eval_dataset=dataset["validation"],      # Full 652 samples

In [19]:
print("train:", len(dataset["train"]))
print("val:", len(dataset["validation"]))
print("test:", len(dataset["test"]))
print("model device:", next(model.parameters()).device)
print("CUDA:", torch.cuda.is_available())

train: 11901
val: 4272
test: 5213
model device: cuda:0
CUDA: True


In [17]:
# Quick test with just 1 step
training_args_test = Seq2SeqTrainingArguments(
    output_dir="./whisper-dysarthria-it",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    num_train_epochs=1,
    fp16=True,
    max_steps=1,  # Just 1 step to test
    logging_steps=1,
    save_strategy="no",
    dataloader_num_workers=0,  # Disable workers for testing
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to="none",
)

trainer_test = Seq2SeqTrainer(
    model=model,
    args=training_args_test,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=processor.tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Starting 1-step test...")
trainer_test.train()
print("Test completed!")

max_steps is given, it will override any value given in num_train_epochs


Starting 1-step test...


  0%|          | 0/1 [00:00<?, ?it/s]e:\Projects\asrfinetune\venv\lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
100%|██████████| 1/1 [00:02<00:00,  2.74s/it]

{'loss': 10.1794, 'grad_norm': inf, 'learning_rate': 5e-06, 'epoch': 0.0}
{'train_runtime': 2.7449, 'train_samples_per_second': 5.829, 'train_steps_per_second': 0.364, 'train_loss': 10.179384231567383, 'epoch': 0.0}
Test completed!


In [20]:
trainer.train()

  0%|          | 0/7440 [00:00<?, ?it/s]e:\Projects\asrfinetune\venv\lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
  0%|          | 25/7440 [00:34<2:45:25,  1.34s/it]

{'loss': 10.1385, 'grad_norm': 248.93896484375, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.03}


  1%|          | 50/7440 [01:08<2:44:22,  1.33s/it]

{'loss': 7.2101, 'grad_norm': 63.955810546875, 'learning_rate': 2.25e-06, 'epoch': 0.07}


  1%|          | 75/7440 [01:41<2:42:45,  1.33s/it]

...

                                                       
 30%|███       | 2232/7440 [2:13:16<1:39:55,  1.15s/it]Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 502

{'eval_loss': 0.056019317358732224, 'eval_wer': 6.398347107438017, 'eval_runtime': 1696.8208, 'eval_samples_per_second': 2.518, 'eval_steps_per_second': 0.315, 'epoch': 3.0}


There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
 30%|███       | 2232/7440 [2:13:19<5:11:06,  3.58s/it]

{'train_runtime': 7999.7277, 'train_samples_per_second': 14.877, 'train_steps_per_second': 0.93, 'train_loss': 0.37551861687640137, 'epoch': 3.0}


TrainOutput(global_step=2232, training_loss=0.37551861687640137, metrics={'train_runtime': 7999.7277, 'train_samples_per_second': 14.877, 'train_steps_per_second': 0.93, 'total_flos': 1.030336454762496e+19, 'train_loss': 0.37551861687640137, 'epoch': 3.0})

In [24]:
import torch

# Check CUDA availability
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

# Check current device
if torch.cuda.is_available():
    print("Current CUDA device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name())
else:
    print("CUDA not available - using CPU")

# Check if model exists and where it is
try:
    print("Model device:", next(model.parameters()).device)
except NameError:
    print("Model not loaded yet")

CUDA available: True
CUDA device count: 1
Current CUDA device: 0
Device name: NVIDIA GeForce RTX 4070 Laptop GPU
Model device: cuda:0


In [21]:
trainer.evaluate(dataset["test"])

100%|██████████| 652/652 [33:46<00:00,  3.11s/it]


{'eval_loss': 0.10100483149290085,
 'eval_wer': 6.914513165952235,
 'eval_runtime': 2030.1906,
 'eval_samples_per_second': 2.568,
 'eval_steps_per_second': 0.321,
 'epoch': 3.0}

In [23]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

sample = dataset["test"][0]

input_features = torch.tensor(sample["input_features"]).unsqueeze(0).to(device)

pred_ids = model.generate(input_features)

print("PRED:", processor.batch_decode(pred_ids, skip_special_tokens=True)[0])

PRED:  there


In [ ]:
import numpy as np
from tqdm import tqdm
import evaluate
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

def prepare_inputs(x):
    x = torch.tensor(np.stack(x), device=device, dtype=torch.float32)
    if next(model.parameters()).dtype == torch.float16:
        x = x.half()
    return x

batch_size = 8

predictions = []
references = []

for i in tqdm(range(0, len(dataset["test"]), batch_size), desc="Inferencing"):
    batch = dataset["test"][i : i + batch_size]

    input_features = prepare_inputs(batch["input_features"])

    with torch.no_grad():
        pred_ids = model.generate(input_features)

    pred_texts = processor.batch_decode(pred_ids, skip_special_tokens=True)
    ref_texts = processor.batch_decode(batch["labels"], skip_special_tokens=True)

    predictions.extend(pred_texts)
    references.extend(ref_texts)

print("Overall WER:", wer_metric.compute_metrics(predictions=predictions, references=references))
print("Overall CER:", cer.compute(predictions=predictions, references=references))

# Print some examples
print("\nSample predictions")
for i in range(8):
    print(f"[{i}] REF: {references[i]}")
    print(f"      PRED: {predictions[i]}")
    print("----")


Inferencing: 100%|██████████| 652/652 [1:24:18<00:00,  7.76s/it]


NameError: name 'wer' is not defined

In [28]:
import evaluate

wer = evaluate.load("wer")

final_wer = wer.compute(
    predictions=predictions,
    references=references
)

print(f"Test WER: {final_wer:.4f}")
print(f"wER (%): {final_wer * 100:.2f}%")

Test WER: 21.3912
wER (%): 2139.12%


In [30]:
# word confusion matrix (top errors)
from collections import Counter
import editdistance

substitutions = Counter()
insertions = Counter()
deletions = Counter()

for ref, pred in zip(references, predictions):
    r_words = ref.split()
    p_words = pred.split()
    # simple alignment via dynamic programming (Levenshtein)
    # use two-pointer greedy for quick approx
    # for exact, use python-levenshtein or similar; here we derive tok-level confusions in matched pairs
    # fallback: keeps wrong/good
    d = editdistance.eval(ref.split(), pred.split())
    # not exact confusion matrix but rough
    # easier: count token-identity mismatch if shapes equal
    for r, p in zip(r_words, p_words):
        if r != p:
            substitutions[(r, p)] += 1

print("Top word substitution errors:")
for (r, p), ct in substitutions.most_common(20):
    print(f"{ct:4d} | {r!r} -> {p!r}")


Top word substitution errors:
  37 | 'vivavoce' -> 'vivavocevai'
  33 | 'chiama' -> 'yama'
  33 | 'chiudi' -> 'you'
  29 | 'stop' -> 'top'
  26 | 'apri' -> 'apriya'
  25 | '3' -> '2'
  22 | 'vai' -> 'vaiya'
  21 | 'principale' -> 'principale.'
  21 | 'chiudi' -> 'aggi'
  19 | 'rubrica' -> 'rubri'
  19 | 'chiamate' -> 'chiamatechiudi'
  18 | 'zero' -> 'there'
  18 | '1' -> '2'
  17 | 'bue' -> 'more'
  17 | 'scesi' -> 'si'
  16 | 'bue' -> 'bueh'
  16 | 'vai' -> 'vaii'
  15 | 'chiudi' -> 'kiudi'
  14 | 'cinque' -> 'si'
  14 | 'aggiungi' -> 'aggi'


In [31]:
eval_out = trainer.evaluate(dataset["test"])
print(eval_out)


100%|██████████| 652/652 [33:26<00:00,  3.08s/it]

{'eval_loss': 0.10100483149290085, 'eval_wer': 6.914513165952235, 'eval_runtime': 2010.0521, 'eval_samples_per_second': 2.593, 'eval_steps_per_second': 0.324, 'epoch': 3.0}


In [3]:
import torch
print(torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

1
0 NVIDIA GeForce RTX 4070 Laptop GPU


In [27]:
import torch
print("Model device:", next(model.parameters()).device)
print("CUDA available:", torch.cuda.is_available())

Model device: cuda:0
CUDA available: True


In [28]:
import torch
print("Model device:", next(model.parameters()).device)
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

Model device: cuda:0
CUDA available: True
Device count: 1


In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Model moved to:", device)

Model moved to: cuda


In [32]:
trainer.save_model("./whisper-dysarthria-it/final")
processor.save_pretrained("./whisper-dysarthria-it/final")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 448, 'suppress_tokens': [1, 2, 7, 8, 9, 10, 14, 25, 26, 27, 28, 29, 31, 58, 59, 60, 61, 62, 63, 90, 91, 92, 93, 359, 503, 522, 542, 873, 893, 902, 918, 922, 931, 1350, 1853, 1982, 2460, 2627, 3246, 3253, 3268, 3536, 3846, 3961, 4183, 4667, 6585, 6647, 7273, 9061, 9383, 10428, 10929, 11938, 12033, 12331, 12562, 13793, 14157, 14635, 15265, 15618, 16553, 16604, 18362, 18956, 20075, 21675, 22520, 26130, 26161, 26435, 28279, 29464, 31650, 32302, 32470, 36865, 42863, 47425, 49870, 50254, 50258, 50360, 50361, 50362], 'begin_suppress_tokens': [220, 50257]}


[]